## StateBackend
Stores files in LangGraph agent state. With a checkpointer, files survive multiple turns on the same thread, but are not shared across different threads.
* Think: conversation-specific scratchpad.

## FilesystemBackend
Stores files on your actual local disk under a configured root directory. Files can be accessed by later agent instances, provided they use the same directory.
* Think: local project folder.

## StoreBackend
Stores files in a LangGraph BaseStore. With a persistent store implementation, files can be shared across threads and survive application restarts.
* StoreBackend is not itself a database. It uses a LangGraph Store supplied to the graph. For a durable setup, that store must itself be backed by persistent storage.
* Think: long-term, application-managed storage.

In [ ]:
## Quick comparison
Backend	                        Lives in	 Cross-thread?	    Survives restart?	    Real file on disk?
1. StateBackend	            LangGraph state	        ❌	                ❌	                    ❌
2. FilesystemBackend	        Your disk	        ✅	                ✅	                    ✅
3. StoreBackend	            A LangGraph store	    ✅	       only w/ persistent store	         ❌

Mental model: the agent never knows the difference. Its tools always speak in file paths — the backend quietly translates every read/write into the right operation underneath. That's what lets you swap backends without touching a line of agent logic.

In [15]:
import os
from pathlib import Path

from dotenv import load_dotenv

from deepagents import create_deep_agent
from deepagents.backends import (
    StateBackend,
    FilesystemBackend,
    StoreBackend,
)

from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore


In [48]:
# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------

load_dotenv()

MODEL_NAME = "qwen3:8b"

# Change this value to test another backend:
# "state", "filesystem", or "store"
BACKEND_TYPE = "filesystem"

# Resolve the project directory
PROJECT_DIR = Path.cwd()

WORKSPACE_DIR = PROJECT_DIR / "agent_workspace"

WORKSPACE_DIR

PosixPath('/Users/vinaykumarranjan/Documents/Langchain/24-DeepAgents/agent_workspace')

In [49]:
# ---------------------------------------------------------
# 2. Initialize the LLM
# ---------------------------------------------------------

model = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
)


In [50]:
# ---------------------------------------------------------
# 3. Initialize LangGraph persistence components
# ---------------------------------------------------------

# Checkpointer stores conversation state by thread_id.
checkpointer = InMemorySaver()

# Store is a key-value store shared across threads.
# This implementation is in-memory, so it is NOT durable
# across Python process restarts.
store = InMemoryStore()

In [51]:
# ---------------------------------------------------------
# 4. Select the backend
# ---------------------------------------------------------

def get_backend():
    if BACKEND_TYPE == "state":

        # Explicitly select the default backend.
        return StateBackend()

    elif BACKEND_TYPE == "filesystem":

        WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

        return FilesystemBackend(
            root_dir=str(WORKSPACE_DIR.resolve()),
            virtual_mode=True,
        )

    elif BACKEND_TYPE == "store":

        return StoreBackend(
            namespace=lambda runtime: ( ##This is a lambda function that dynamically returns a namespace (a tuple of strings) for the StoreBackend.
                "deepagents-backend-demo", ## Application or project
                "student-001", ##User or student identifier
            )
        )

        # def get_namespace(runtime): ## same interpretation of above lambda function
        #     return (
        #         "deepagents-backend-demo",
        #         "student-001",
        #     )

    else:
        raise ValueError(
            f"Unsupported backend: {BACKEND_TYPE}"
        )


backend = get_backend()

backend


## What is a namespace?
Think of a LangGraph Store as a database containing many records.

A namespace acts like a logical grouping or folder that organizes those records.


In [ ]:
LangGraph Store
│
├── deepagents-backend-demo
│   │
│   └── student-001
│       ├── research/ai_notes.md
│       └── research/ml_notes.md
│
└── customer-support
    │
    └── customer-101
        ├── preferences.md
        └── history.md

In [53]:
# ---------------------------------------------------------
# 5. Create the Deep Agent
# ---------------------------------------------------------

agent = create_deep_agent(
    model=model,
    backend=backend,
    checkpointer=checkpointer,
    store=store if BACKEND_TYPE == "store" else None,
    system_prompt="""
    You are a helpful research assistant.

    You have access to filesystem tools.

    Rules:
    1. Use write_file when asked to create a file.
    2. Use read_file when asked to read a file.
    3. Never claim a file was saved unless you actually
       called write_file.
    4. If a file does not exist, clearly report that.
    5. Do not invent file contents.
    """,
)


In [54]:
# ---------------------------------------------------------
# 6. Helper function to invoke the agent
# ---------------------------------------------------------

def run_agent(question: str, thread_id: str):

    print("\n" + "=" * 65)
    print(f"THREAD ID: {thread_id}")
    print("=" * 65)

    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        },
        config={
            "configurable": {
                "thread_id": thread_id
            }
        },
    )

    final_message = result["messages"][-1]

    print("\nAGENT RESPONSE:")
    print(final_message.content)

    return result


In [ ]:
Setting	                             What you'll observe
"state"	                        Thread A can read the file. Thread B cannot.
"filesystem"	                Both Thread A and Thread B can read the file.
"store"	                        Both Thread A and Thread B can read the file.

In [55]:
# ---------------------------------------------------------
# 7. Experiment 1: Write a file in Thread A
# ---------------------------------------------------------

run_agent(
    question="""
    Create a file named /research/ai_notes.md.

    Write the following content:

    # Artificial Intelligence

    Artificial Intelligence enables machines to perform
    tasks that normally require human intelligence.

    # Machine Learning

    Machine Learning is a subset of AI that enables
    systems to learn patterns from data.

    Confirm after creating the file.
    """,
    thread_id="thread-A",
)


THREAD ID: thread-A

AGENT RESPONSE:
The file `/research/ai_notes.md` has been created successfully with the provided content. 

Would you like to review the contents, add more information, or perform another task?


{'messages': [HumanMessage(content='\n    Create a file named /research/ai_notes.md.\n\n    Write the following content:\n\n    # Artificial Intelligence\n\n    Artificial Intelligence enables machines to perform\n    tasks that normally require human intelligence.\n\n    # Machine Learning\n\n    Machine Learning is a subset of AI that enables\n    systems to learn patterns from data.\n\n    Confirm after creating the file.\n    ', additional_kwargs={}, response_metadata={}, id='b5b45d0c-b9ab-4e0d-a9af-31b1655a04cc'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-11-03T05:17:33.516882Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9881709541, 'load_duration': 5151458, 'prompt_eval_count': 2424, 'prompt_eval_duration': 492430000, 'eval_count': 195, 'eval_duration': 9353645000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a19fd7-352e-7722-a111-2e0244e1f029-0', tool_calls=[{'n

In [56]:
# ---------------------------------------------------------
# 8. Experiment 2: Read the file in the SAME thread
# ---------------------------------------------------------

run_agent(
    question="""
    Read /research/ai_notes.md and summarize its contents.
    """,
    thread_id="thread-A",
)


THREAD ID: thread-A

AGENT RESPONSE:
The file `/research/ai_notes.md` contains two sections:

1. **Artificial Intelligence**:  
   Defines AI as the ability of machines to perform tasks requiring human intelligence.

2. **Machine Learning**:  
   Describes it as a subset of AI focused on enabling systems to learn patterns from data.

Would you like to expand on either of these topics or modify the file?


{'messages': [HumanMessage(content='\n    Create a file named /research/ai_notes.md.\n\n    Write the following content:\n\n    # Artificial Intelligence\n\n    Artificial Intelligence enables machines to perform\n    tasks that normally require human intelligence.\n\n    # Machine Learning\n\n    Machine Learning is a subset of AI that enables\n    systems to learn patterns from data.\n\n    Confirm after creating the file.\n    ', additional_kwargs={}, response_metadata={}, id='b5b45d0c-b9ab-4e0d-a9af-31b1655a04cc'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-11-03T05:17:33.516882Z', 'done': True, 'done_reason': 'stop', 'total_duration': 9881709541, 'load_duration': 5151458, 'prompt_eval_count': 2424, 'prompt_eval_duration': 492430000, 'eval_count': 195, 'eval_duration': 9353645000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a19fd7-352e-7722-a111-2e0244e1f029-0', tool_calls=[{'n

In [57]:
# ---------------------------------------------------------
# 9. Experiment 3: Read the file in a DIFFERENT thread
# ---------------------------------------------------------

run_agent(
    question="""
    Read /research/ai_notes.md.

    If it does not exist, tell me that the file is unavailable.
    Do not recreate it.
    """,
    thread_id="thread-B",
)



THREAD ID: thread-B

AGENT RESPONSE:
The file `/research/ai_notes.md` contains the following content:

```
# Artificial Intelligence

Artificial Intelligence enables machines to perform
tasks that normally require human intelligence.

# Machine Learning

Machine Learning is a subset of AI that enables
systems to learn patterns from data.
```

Let me know if you'd like to edit, analyze, or expand on this content!


{'messages': [HumanMessage(content='\n    Read /research/ai_notes.md.\n\n    If it does not exist, tell me that the file is unavailable.\n    Do not recreate it.\n    ', additional_kwargs={}, response_metadata={}, id='43dd41f6-532c-41ec-882d-812fc6d4a185'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-11-03T05:18:35.860787Z', 'done': True, 'done_reason': 'stop', 'total_duration': 8287046625, 'load_duration': 12092708, 'prompt_eval_count': 2384, 'prompt_eval_duration': 317748000, 'eval_count': 169, 'eval_duration': 7927168000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a19fd8-2ef1-71f2-af2f-e87fff3e7e5a-0', tool_calls=[{'name': 'read_file', 'args': {'file_path': '/research/ai_notes.md'}, 'id': '867884dc-a1c3-4650-919d-667a7a3874c8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2384, 'output_tokens': 169, 'total_tokens': 2553}),
  ToolMessage(content='

In [58]:
# ---------------------------------------------------------
# 10. Inspect actual disk storage
# ---------------------------------------------------------

if BACKEND_TYPE == "filesystem":

    print("\n" + "=" * 65)
    print("FILES CREATED ON YOUR LOCAL DISK")
    print("=" * 65)

    for path in WORKSPACE_DIR.rglob("*"):
        if path.is_file():
            print(path)



FILES CREATED ON YOUR LOCAL DISK
/Users/vinaykumarranjan/Documents/Langchain/24-DeepAgents/agent_workspace/research/ai_notes.md
